# 06. Complement Technique (Hashing)

The complement technique is a hashing pattern used when a problem asks for values that combine with the current value to reach a target.

For a current value `x` and target `T`, the required complement is `T - x`.

**Core question:** Have I already seen the value that completes the current value?

## 1. What is a Complement?

A complement is the missing value needed to satisfy a target condition.

For a sum problem:

`x + y = target`

Rearrange:

`y = target - x`

So, while processing `x`, we search for `target - x`.

Example: target = 10 and current value = 4. The complement is 6 because `4 + 6 = 10`.

In [1]:
target = 10
current = 4
complement = target - current

print(complement)  # 6

6


## 2. Why Use Hashing?

Brute force checks every pair using two loops: O(n²) time.

Hashing remembers previously seen values. For each current value, we calculate its complement and check whether it has already appeared.

- Average time: O(n)
- Extra space: O(n)

This is a time-space trade-off: we use memory to avoid repeated pair comparisons.

## 3. Pattern 1 — Two Sum (Return Indices)

Given an array and a target, return indices of two different elements whose sum equals the target.

Algorithm:

1. Create an empty dictionary `seen` mapping value to index.
2. Scan the array from left to right.
3. Calculate `complement = target - num`.
4. If complement is in `seen`, return its stored index and the current index.
5. Otherwise store `seen[num] = i`.

**Why check before inserting?** It prevents using the current element twice. It also correctly handles `[3, 3]` with target 6: the first 3 is stored, and the second 3 finds it.

In [2]:
def two_sum(nums, target):
    seen = {}  # value -> index

    for i, num in enumerate(nums):
        complement = target - num

        if complement in seen:
            return [seen[complement], i]

        seen[num] = i

    return []


print(two_sum([2, 7, 11, 15], 9))  # [0, 1]
print(two_sum([3, 3], 6))          # [0, 1]

[0, 1]
[0, 1]


## 4. Dry Run — Two Sum

Input: `nums = [2, 7, 11, 15]`, `target = 9`

| Index | Current | Complement | Seen before check | Action |
|---:|---:|---:|---|---|
| 0 | 2 | 7 | `{}` | Store `2: 0` |
| 1 | 7 | 2 | `{2: 0}` | Found 2; return `[0, 1]` |

Notice that the dictionary contains only earlier elements. This is what guarantees two different indices.

## 5. Pattern 2 — Two Sum (Return True or False)

If the problem only asks whether a pair exists, a set is enough. We do not need indices.

For each number:

1. Calculate its complement.
2. Check whether the complement is in the set of previous values.
3. If yes, return True.
4. Otherwise, add the current number.

Average time O(n), extra space O(n).

In [3]:
def has_pair_with_sum(nums, target):
    seen = set()

    for num in nums:
        complement = target - num
        if complement in seen:
            return True
        seen.add(num)

    return False


print(has_pair_with_sum([1, 4, 6, 8], 10))  # True
print(has_pair_with_sum([1, 2, 3], 10))     # False

True
False


## 6. Pattern 3 — Count Pairs with a Target Sum

Sometimes the question asks for the number of index pairs, not just whether a pair exists.

For each current value `x`, the number of earlier values equal to `target - x` tells us how many new pairs end at this index.

Store frequencies of previously seen values.

Important: this counts index pairs. Duplicate values at different indices can create multiple valid pairs.

In [4]:
def count_pairs(nums, target):
    freq = {}
    pairs = 0

    for num in nums:
        complement = target - num
        pairs += freq.get(complement, 0)
        freq[num] = freq.get(num, 0) + 1

    return pairs


print(count_pairs([1, 5, 7, -1, 5], 6))  # 3

3


## 7. Why Update the Frequency After Counting?

In the pair-counting algorithm, first add the number of previously seen complements, then record the current number.

This ensures the current element cannot pair with itself.

Example: `nums = [3]`, target = 6. The answer must be 0, not 1.

If the current value were added before counting, it could incorrectly match itself.

In [5]:
print(count_pairs([3], 6))       # 0
print(count_pairs([3, 3], 6))     # 1
print(count_pairs([3, 3, 3], 6))  # 3

0
1
3


## 8. Pattern 4 — Count Pairs with a Difference

For an absolute difference `k`, a current value `x` can pair with `x - k` or `x + k`.

But be careful: problems differ in what they count:

- Unique value pairs: each distinct pair counted once.
- Index pairs: every pair of positions counted.
- Pairs with duplicates: multiplicity matters.

Always clarify the required meaning before choosing a set or frequency map.

## 9. Pattern 5 — General Complement Equations

The same idea works whenever a condition can be rearranged into a required lookup.

Examples:

- Sum: `x + y = T` → find `T - x`.
- Difference: `x - y = D` → find `x - D`.
- Product: `x * y = P` → find `P / x` when division and zero cases are handled.
- XOR: `x ^ y = T` → find `T ^ x`.

Do not apply the formula blindly. Check integer constraints, zero, duplicates, signs, and whether the operation is reversible.

## 10. Hashing vs Sorting + Two Pointers

For pair-sum problems, two common approaches are:

**Hashing:**
- Usually O(n) average time.
- O(n) extra space.
- Works without sorting and can preserve original indices naturally.

**Sorting + two pointers:**
- O(n log n) time including sorting.
- Often O(1) or O(log n) auxiliary space depending on sorting and implementation.
- Sorting changes order, so preserving original indices may require storing `(value, index)` pairs.

Choose based on constraints, required output, and space limits.

In [6]:
def has_pair_sorted(nums, target):
    arr = sorted(nums)
    left, right = 0, len(arr) - 1

    while left < right:
        total = arr[left] + arr[right]
        if total == target:
            return True
        if total < target:
            left += 1
        else:
            right -= 1

    return False


print(has_pair_sorted([8, 1, 4, 6], 10))  # True

True


## 11. Common Mistakes

1. Checking after inserting the current value in Two Sum, which may reuse one index.
2. Using a set when the problem asks for the number of index pairs.
3. Forgetting duplicate values can represent different indices.
4. Returning values when the problem asks for indices.
5. Assuming one pair-counting formula handles unique pairs and index pairs identically.
6. Ignoring negative numbers or zero.
7. Forgetting the extra O(n) memory used by hashing.
8. Failing to state average-case complexity.

## 12. Interview Recognition Checklist

Think about complements when:

- A pair must sum to a target.
- A pair must reach a target difference or XOR.
- A brute-force solution checks all pairs.
- You can rearrange the equation to calculate the missing value.
- Earlier elements can be stored for quick lookup.

Before coding, ask:

1. What equation must the pair satisfy?
2. What is the complement formula?
3. Do I need existence, indices, values, or a count of pairs?
4. Should I use a set, index map, or frequency map?
5. Should I check before updating the data structure?

## 13. Mini Exercises

Try these without looking up solutions:

1. Return indices of a pair whose sum equals target.
2. Return True if any pair sums to target.
3. Count all index pairs whose sum equals target.
4. Explain why the current value is inserted after checking its complement.
5. Solve pair sum using sorting and two pointers.
6. Explain when hashing is preferable to sorting.
7. For `x ^ y = target`, derive the complement expression.

**Challenge:** Given an integer array and a target, count the number of index pairs `(i, j)` such that `i < j` and `nums[i] + nums[j] == target`. Test duplicates and negative values.

In [7]:
def count_target_pairs(nums, target):
    # Complete this using a frequency map.
    freq = {}
    pairs = 0

    for num in nums:
        # 1. Find the complement.
        # 2. Add the number of earlier complements.
        # 3. Record the current number.
        pass

    return pairs


# Expected: 3
print(count_target_pairs([1, 5, 7, -1, 5], 6))

0


## 14. Complexity Summary and Reusable Templates

| Pattern | Average time | Extra space |
|---|---:|---:|
| Two Sum with index map | O(n) | O(n) |
| Pair existence with set | O(n) | O(n) |
| Count index pairs | O(n) | O(k) |
| Sorting + two pointers | O(n log n) | Depends on sorting |

### Reusable templates

```python
# Return indices
seen = {}
for i, x in enumerate(nums):
    need = target - x
    if need in seen:
        return [seen[need], i]
    seen[x] = i

# Pair existence
seen = set()
for x in nums:
    need = target - x
    if need in seen:
        return True
    seen.add(x)

# Count index pairs
freq = {}
pairs = 0
for x in nums:
    need = target - x
    pairs += freq.get(need, 0)
    freq[x] = freq.get(x, 0) + 1
```

**Key idea:** derive the missing value from the equation, then use hashing to check whether it has already appeared.

**Next concept:** Duplicate Detection — identifying repeated values and understanding unique values versus repeated index occurrences.